# Interactive RSI Crossover Visualizer (Plotly)
# Exploring: RSI(3) crossing RSI(14) with 55-WMA Regime Filter
# max stocks: 5
# default view: 6 months

In [1]:
# 1. Imports and Setup
import pandas as pd
import numpy as np
import os
import glob
import random
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML
import warnings

warnings.filterwarnings('ignore')
display(HTML("<style>.container { width:100% !important; }</style>"))

DATA_DIR = r'D:\0dot1_Aug_2016_master\data\mstock_mtf_daily_data'


In [2]:
# 2. Data Loading & Calculations
def calc_rsi(series, period):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def calc_wma(series, period):
    weights = np.arange(1, period + 1)
    return series.rolling(period).apply(lambda prices: np.dot(prices, weights) / weights.sum(), raw=True)

all_data = []
files = glob.glob(os.path.join(DATA_DIR, '*.csv'))
random.seed(42) 
random.shuffle(files)

loaded = 0
for file_path in files:
    if loaded >= 5: 
        break
    symbol = os.path.basename(file_path).replace('.csv', '')
    try:
        df = pd.read_csv(file_path).dropna(subset=['Close'])
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.sort_values('Date').reset_index(drop=True)
        
        # Need history
        if len(df) < 1100: 
            continue
            
        latest_price = df['Close'].iloc[-1]
        if not (90 <= latest_price <= 600):
            continue
            
        # Indicators
        df['WMA_55'] = calc_wma(df['Close'], 275) # Approx 55 weekly
        df['RSI_3'] = calc_rsi(df['Close'], 3)
        df['RSI_14'] = calc_rsi(df['Close'], 14)
        df['RSI_Ratio'] = df['RSI_3'] / (df['RSI_14'] + 0.0001)
        
        # Keep 3 years of data (approx 750 days) so you can scroll back
        df = df.tail(750).reset_index(drop=True)
        
        # Find Crossover Entries: Price > WMA_55 AND RSI_14 < 50 AND Ratio crosses above 1.0
        df['Crossover_Signal'] = (df['Close'] > df['WMA_55']) & (df['RSI_14'] < 50) & (df['RSI_Ratio'] >= 1.0) & (df['RSI_Ratio'].shift(1) < 1.0)
        
        df['Symbol'] = symbol
        all_data.append(df)
        loaded += 1
    except Exception as e:
        pass

data = pd.concat(all_data, ignore_index=True)
print(f"Loaded {len(data['Symbol'].unique())} stocks for Plotly analysis.")


Loaded 5 stocks for Plotly analysis.


In [3]:
# 3. Interactive Plotly Charts
symbols = data['Symbol'].unique()

for symbol in symbols:
    df = data[data['Symbol'] == symbol].reset_index(drop=True)
    
    # Define default 6-month view range
    end_date = df['Date'].max()
    start_date_6m = end_date - pd.Timedelta(days=180)
    
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, 
                        vertical_spacing=0.03, 
                        row_heights=[0.5, 0.25, 0.25])
    
    # --- Row 1: Price and WMA ---
    fig.add_trace(go.Scatter(x=df['Date'], y=df['Close'], mode='lines', name='Close Price', line=dict(color='black', width=1.5)), row=1, col=1)
    fig.add_trace(go.Scatter(x=df['Date'], y=df['WMA_55'], mode='lines', name='55-WMA', line=dict(color='orange', width=2.5)), row=1, col=1)
    
    # Mark Crossover Signals on Price Chart
    signals = df[df['Crossover_Signal']]
    fig.add_trace(go.Scatter(x=signals['Date'], y=signals['Close'], mode='markers', name='Crossover Buy Signal', 
                             marker=dict(symbol='triangle-up', color='lime', size=12, line=dict(color='black', width=1))), row=1, col=1)

    # --- Row 2: RSI(3) vs RSI(14) ---
    fig.add_trace(go.Scatter(x=df['Date'], y=df['RSI_3'], mode='lines', name='RSI (3)', line=dict(color='dodgerblue', width=1.5)), row=2, col=1)
    fig.add_trace(go.Scatter(x=df['Date'], y=df['RSI_14'], mode='lines', name='RSI (14)', line=dict(color='magenta', width=1.5, dash='dot')), row=2, col=1)
    # 50 Line
    fig.add_hline(y=50, line_dash="dash", line_color="gray", line_width=1, row=2, col=1)
    
    # --- Row 3: RSI Ratio ---
    fig.add_trace(go.Scatter(x=df['Date'], y=df['RSI_Ratio'], mode='lines', name='Ratio (3/14)', line=dict(color='purple', width=1.5)), row=3, col=1)
    fig.add_hline(y=1.0, line_dash="dash", line_color="red", line_width=2, row=3, col=1)

    # Layout configuration
    fig.update_layout(
        title=f"{symbol} - Interactive RSI Crossover Explorer",
        height=900,
        width=1400,
        hovermode='x unified',
        showlegend=True,
        plot_bgcolor='white',
        hoverdistance=100
    )
    
    # Grid styling
    fig.update_xaxes(
        showgrid=True, 
        gridwidth=1, 
        gridcolor='lightgray',
        range=[start_date_6m, end_date] # Apply range to ALL x-axes to preserve sync
    )
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')

    
    fig.show()
